# Callable Objects: An Advanced, Step-by-Step Problem Workshop

**Topic:** Python objects that implement `__call__` — using state, time, control flow, validation, parsing, orchestration, and operator overloading.

This is a **new set of worked problems**, written in the exploratory style of the supplied lesson: short experiments and lots of explanation between small, executable code cells. The lesson begins with simple `__call__` objects and gradually introduces partial application, cache factories, and profiling decorators. Here we use that same *teaching rhythm* while tackling different engineering problems.

**How to study:** Run the notebook from top to bottom. For each problem, read the contract, predict each small experiment, implement the missing idea mentally, compare the worked solution, then run the assertions. All exercise solutions are included; no code is intentionally left broken.

**Requirements:** Python 3.10+; standard library only. No network, randomness without a seed, real sleeps, external datasets, or third-party dependencies.

**Contents:** 13 independent advanced problems, each broken into logical stages, plus a final design checklist. Code cells contain examples and automated assertions; Markdown cells explain decisions and limitations.

## A tiny test toolbox

To explore failure paths without stopping the notebook, use `expect_raises`. This context manager is *not* a testing framework; it just makes exceptional behavior visible. We will also use a manually controlled clock so time-related behavior is reproducible.

In [1]:
from contextlib import contextmanager
from collections import OrderedDict, deque
from dataclasses import dataclass
from functools import wraps
from inspect import signature
from math import isfinite, sqrt
from typing import Any

@contextmanager
def expect_raises(error_type):
    try:
        yield
    except error_type as exc:
        print(f"Expected {type(exc).__name__}: {exc}")
    else:
        raise AssertionError(f"Expected {error_type.__name__} to be raised")

class ManualClock:
    """Time advances only when our test explicitly advances it."""
    def __init__(self, initial=0.0):
        self.now = float(initial)

    def __call__(self):
        return self.now

    def advance(self, seconds):
        if seconds < 0:
            raise ValueError("cannot travel backward using advance")
        self.now += seconds

clock_demo = ManualClock(10)
assert clock_demo() == 10
clock_demo.advance(2.5)
assert clock_demo() == 12.5
print("Test toolbox ready")

Test toolbox ready


---

# Problem 01 — A callable traffic-light controller

**Challenge.** Implement a command processor whose `__call__` transitions among states, records successful commands, and never corrupts state on invalid input.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Define the contract before defining the class

A callable object can represent a *stateful machine*, not just a mathematical function. Calling `controller("next")` should advance `RED → GREEN → YELLOW → RED`. Calling `controller("status")` should report the state without changing history. `"reset"` should set `RED` and record the change. Unknown commands must raise `ValueError` **before** changing anything.

We should not put history in a class attribute: different controllers need independent logs.

In [2]:
TRANSITIONS = {"RED": "GREEN", "GREEN": "YELLOW", "YELLOW": "RED"}
assert TRANSITIONS[TRANSITIONS[TRANSITIONS["RED"]]] == "RED"
print("Cycle:", " → ".join(["RED", "GREEN", "YELLOW", "RED"]))

Cycle: RED → GREEN → YELLOW → RED


### Step 2 — A common state-leak bug

Mutable class attributes are shared by instances. The following is a deliberately *incorrect experiment*, not the implementation we will use. Observe the surprising behavior before we fix it.

In [3]:
class IncorrectController:
    history = []
    def __call__(self, item):
        self.history.append(item)

one, two = IncorrectController(), IncorrectController()
one("from one")
print("Second object sees:", two.history)
assert two.history == ["from one"]

Second object sees: ['from one']


### Step 3 — Worked implementation

We allocate state in `__init__`, validate the command first, use a transition table rather than nested conditionals, and expose a **snapshot** (tuple) so outside code cannot append directly to the internal list. `__call__` returns the current state for every valid command.

In [4]:
class TrafficLight:
    TRANSITIONS = {"RED": "GREEN", "GREEN": "YELLOW", "YELLOW": "RED"}
    COMMANDS = frozenset({"next", "reset", "status"})

    def __init__(self):
        self._state = "RED"
        self._history = []

    @property
    def state(self):
        return self._state

    @property
    def history(self):
        return tuple(self._history)

    def __call__(self, command):
        if command not in self.COMMANDS:
            raise ValueError(f"unknown command: {command!r}")
        if command == "status":
            return self._state
        if command == "reset":
            self._state = "RED"
        else:
            self._state = self.TRANSITIONS[self._state]
        self._history.append((command, self._state))
        return self._state

### Step 4 — Trace one object

The first call is the same syntax as calling a function, but the `TrafficLight` instance remembers earlier calls. Verify both the return value and state changes.

In [5]:
light = TrafficLight()
assert callable(light)
assert light("status") == "RED"
assert light("next") == "GREEN"
assert light("next") == "YELLOW"
assert light("next") == "RED"
assert light.history == (("next", "GREEN"), ("next", "YELLOW"), ("next", "RED"))
print("State:", light.state, "History:", light.history)

State: RED History: (('next', 'GREEN'), ('next', 'YELLOW'), ('next', 'RED'))


### Step 5 — Invalid inputs and state isolation

Error handling is part of the contract. Compare snapshots taken before and after a failed call. Then create a second object to check that it has no shared history.

In [6]:
before = (light.state, light.history)
with expect_raises(ValueError):
    light("skip")
assert (light.state, light.history) == before
other_light = TrafficLight()
assert other_light.history == () and other_light.state == "RED"
assert light("reset") == "RED"
assert light.history[-1] == ("reset", "RED")
print("Problem 1: all checks passed")

Expected ValueError: unknown command: 'skip'
Problem 1: all checks passed


### What we learned

`__call__` is a natural interface for a small command processor. Explicit transition tables and per-instance state simplify reasoning. An immutable *view* of history protects the invariant; remember that a tuple only prevents top-level mutation of the stored entries, so our entries are tuples of strings.

---

# Problem 02 — Online statistics without retaining the dataset

**Challenge.** Build a numerical callable that consumes observations one by one and supports a numerically stable mean, sample variance, a merge operation, and resets.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Why the obvious approach is undesirable

Saving every input and recomputing the mean and variance would use memory proportional to the number of samples. The formula `E[x²] − E[x]²` can also suffer catastrophic cancellation when values are very large but their spread is tiny.

We will use Welford’s online update. Maintain sample count `n`, mean `μ`, and sum of squared deviations `M2`. For each new `x`: `delta=x−μ`, update `μ += delta/n`, then `M2 += delta*(x−μ_new)`. Sample variance is `M2/(n−1)` when `n >= 2`.

In [7]:
data_example = (2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0)
expected_mean = sum(data_example) / len(data_example)
expected_sample_variance = sum((x-expected_mean)**2 for x in data_example) / (len(data_example)-1)
print("Reference:", expected_mean, expected_sample_variance)

Reference: 5.0 4.571428571428571


### Step 2 — Decide on the call interface

One invocation of `RunningStats(x)` will consume *one* finite real observation and return the updated mean. Separate read-only properties expose the count, variance, and standard deviation. The absence of samples makes the mean undefined; we raise `ValueError` instead of inventing `0.0`.

In [8]:
class RunningStats:
    def __init__(self):
        self.reset()

    def reset(self):
        self._n = 0
        self._mean = 0.0
        self._m2 = 0.0

    @property
    def count(self):
        return self._n

    @property
    def mean(self):
        if self._n == 0:
            raise ValueError("mean is undefined for zero observations")
        return self._mean

    @property
    def sample_variance(self):
        if self._n < 2:
            raise ValueError("sample variance needs at least two observations")
        return self._m2 / (self._n - 1)

    @property
    def sample_std(self):
        return sqrt(self.sample_variance)

    def __call__(self, x):
        if isinstance(x, bool) or not isinstance(x, (int, float)):
            raise TypeError("observation must be a real number")
        if not isfinite(x):
            raise ValueError("observation must be finite")
        self._n += 1
        delta = x - self._mean
        self._mean += delta / self._n
        self._m2 += delta * (x - self._mean)
        return self._mean

### Step 3 — Run one observation at a time

The accumulated object updates without keeping `data_example`. When we have fewer than two observations, sample variance is mathematically undefined; test that failure path too.

In [9]:
stats = RunningStats()
with expect_raises(ValueError):
    _ = stats.mean
assert stats(2.0) == 2.0
with expect_raises(ValueError):
    _ = stats.sample_variance
for observation in data_example[1:]:
    stats(observation)
assert stats.count == 8
assert abs(stats.mean - expected_mean) < 1e-12
assert abs(stats.sample_variance - expected_sample_variance) < 1e-12
print("n:", stats.count, "mean:", stats.mean, "sample variance:", stats.sample_variance)

Expected ValueError: mean is undefined for zero observations
Expected ValueError: sample variance needs at least two observations
n: 8 mean: 5.0 sample variance: 4.571428571428571


### Step 4 — Advanced requirement: merge independent streams

Imagine two sensors processing halves of a dataset. We want to merge their summaries without replaying the observations. For summaries `A` and `B`, let `delta=mean_B−mean_A` and `n=n_A+n_B`; then:

`mean = mean_A + delta*n_B/n`

`M2 = M2_A + M2_B + delta²*n_A*n_B/n`.

Merging with an empty summary is a no-op; merging a nonempty summary *into* an empty one copies its state. Reject non-`RunningStats` objects explicitly.

In [10]:
def merge_stats(self, other):
    if not isinstance(other, RunningStats):
        raise TypeError("can only merge RunningStats")
    if other.count == 0:
        return self
    if self.count == 0:
        self._n, self._mean, self._m2 = other._n, other._mean, other._m2
        return self
    delta = other._mean - self._mean
    n = self._n + other._n
    self._m2 += other._m2 + delta**2 * self._n * other._n / n
    self._mean += delta * other._n / n
    self._n = n
    return self

RunningStats.merge = merge_stats

### Step 5 — Prove that the merge reproduces sequential processing

We deliberately use two independently constructed objects. Their merge should agree with the one-pass reference up to floating-point rounding. Checking only the mean would miss an error in the `M2` correction term.

In [11]:
left, right = RunningStats(), RunningStats()
for x in data_example[:3]:
    left(x)
for x in data_example[3:]:
    right(x)
assert left.merge(right) is left
assert left.count == stats.count
assert abs(left.mean - stats.mean) < 1e-12
assert abs(left.sample_variance - stats.sample_variance) < 1e-12
empty = RunningStats()
left.merge(empty)
assert left.count == 8
empty.merge(right)
assert empty.count == 5
with expect_raises(TypeError):
    stats("oops")
with expect_raises(ValueError):
    stats(float("nan"))
assert stats.count == 8
stats.reset()
assert stats.count == 0
print("Problem 2: merge, validation, and reset passed")

Expected TypeError: observation must be a real number
Expected ValueError: observation must be finite
Problem 2: merge, validation, and reset passed


### What we learned

The callable syntax can model an *online accumulator*: each call has a small, documented state transition and needs constant memory. Stateful interfaces become much easier to use when queries, updates, resets, and error cases are clearly separated.

---

# Problem 03 — A circuit breaker with a clock you control

**Challenge.** Wrap an unreliable function in a callable that opens after repeated failures, refuses calls during a cooldown, and permits a single recovery probe.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Draw the state machine in words

We will use three modes: `CLOSED` (ordinary calls), `OPEN` (reject calls without invoking the wrapped function), and `HALF_OPEN` (one probe after cooldown). In `CLOSED`, reaching `failure_threshold` consecutive exceptions opens the circuit. A success resets the counter. In `HALF_OPEN`, success closes the circuit; failure reopens it and restarts the cooldown.

This model assumes *sequential* use. Concurrent use would require synchronization to ensure only one recovery probe is admitted.

In [12]:
class CircuitOpenError(RuntimeError):
    pass

circuit_clock = ManualClock(100)
print("Time:", circuit_clock(), "— no real sleeps required")

Time: 100.0 — no real sleeps required


### Step 2 — Constructor validation prevents impossible configurations

A zero failure threshold or negative cooldown would make the semantics ambiguous. We require a callable wrapped operation, a positive integer threshold, and a finite nonnegative cooldown. The clock is also callable, so our tests can replace wall-clock time.

In [13]:
class CircuitBreaker:
    def __init__(self, fn, *, failure_threshold=2, cooldown=5.0, clock=None):
        if not callable(fn):
            raise TypeError("fn must be callable")
        if type(failure_threshold) is not int or failure_threshold < 1:
            raise ValueError("failure_threshold must be a positive integer")
        if isinstance(cooldown, bool) or not isinstance(cooldown, (int, float)) or not isfinite(cooldown) or cooldown < 0:
            raise ValueError("cooldown must be finite and nonnegative")
        if clock is None:
            from time import monotonic
            clock = monotonic
        if not callable(clock):
            raise TypeError("clock must be callable")
        self.fn = fn
        self.failure_threshold = failure_threshold
        self.cooldown = float(cooldown)
        self.clock = clock
        self.state = "CLOSED"
        self.consecutive_failures = 0
        self.opened_at = None

    def __call__(self, *args, **kwargs):
        now = self.clock()
        if self.state == "OPEN":
            if now - self.opened_at < self.cooldown:
                raise CircuitOpenError("circuit is cooling down")
            self.state = "HALF_OPEN"
        try:
            result = self.fn(*args, **kwargs)
        except Exception:
            if self.state == "HALF_OPEN":
                self._open(now)
            else:
                self.consecutive_failures += 1
                if self.consecutive_failures >= self.failure_threshold:
                    self._open(now)
            raise
        else:
            self.state = "CLOSED"
            self.consecutive_failures = 0
            self.opened_at = None
            return result

    def _open(self, now):
        self.state = "OPEN"
        self.consecutive_failures = 0
        self.opened_at = now

### Step 3 — Build an unreliable function whose behavior we can predict

Injecting a deterministic fake service is more informative than making random network calls. Count *actual* underlying invocations so we can prove an open circuit rejects work rather than merely raising an exception afterward.

In [14]:
class ScriptedService:
    def __init__(self, outcomes):
        self.outcomes = iter(outcomes)
        self.calls = 0

    def __call__(self, payload):
        self.calls += 1
        outcome = next(self.outcomes)
        if outcome == "FAIL":
            raise OSError("simulated outage")
        return f"processed:{payload}"

service = ScriptedService(["FAIL", "FAIL", "FAIL", "OK", "OK"])
breaker = CircuitBreaker(service, failure_threshold=2, cooldown=5, clock=circuit_clock)

### Step 4 — Drive CLOSED → OPEN

Two consecutive service failures open the circuit. Calls made *during* the cooldown should raise `CircuitOpenError` and should not call the service. This difference is the purpose of a circuit breaker.

In [15]:
with expect_raises(OSError):
    breaker("a")
assert breaker.state == "CLOSED"
with expect_raises(OSError):
    breaker("b")
assert breaker.state == "OPEN" and service.calls == 2
with expect_raises(CircuitOpenError):
    breaker("blocked")
assert service.calls == 2
print("OPEN; real service calls:", service.calls)

Expected OSError: simulated outage
Expected OSError: simulated outage
Expected CircuitOpenError: circuit is cooling down
OPEN; real service calls: 2


### Step 5 — Probe at the exact cooldown boundary

At precisely `opened_at + cooldown`, the circuit permits a probe. Here the first probe fails, so the circuit reopens with a *new* start time. The next probe succeeds and closes the circuit.

In [16]:
circuit_clock.advance(5)
with expect_raises(OSError):
    breaker("probe-1")
assert breaker.state == "OPEN" and service.calls == 3
with expect_raises(CircuitOpenError):
    breaker("still-blocked")
circuit_clock.advance(5)
assert breaker("probe-2") == "processed:probe-2"
assert breaker.state == "CLOSED" and service.calls == 4
assert breaker("normal") == "processed:normal"
assert service.calls == 5
print("Problem 3: recovery path passed")

Expected OSError: simulated outage
Expected CircuitOpenError: circuit is cooling down
Problem 3: recovery path passed


### Design note — What this version deliberately does not claim

Catching `Exception` treats ordinary application exceptions as failures, while allowing `KeyboardInterrupt` and `SystemExit` to propagate. Real systems often count *only selected* failure types and use thread-safe half-open admission. This exercise focuses on a fully specified sequential contract, not a production-ready distributed breaker.

---

# Problem 04 — A token-bucket admission controller

**Challenge.** Build a callable rate limiter that accepts a cost per request, refills tokens using elapsed time, and rejects operations without running them when capacity is exhausted.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Understand the arithmetic

A token bucket starts with `capacity` tokens. After `elapsed` seconds, its balance becomes `min(capacity, old_tokens + elapsed * rate)`. A request costing `cost` is accepted only if enough tokens remain; acceptance removes `cost` tokens.

**Crucial ordering:** check admission *before* calling the wrapped operation. Otherwise a rejected operation could still create side effects.

In [17]:
rate_clock = ManualClock(0)
print("A bucket with capacity=3 and rate=0.5 refills one token every 2 seconds.")

A bucket with capacity=3 and rate=0.5 refills one token every 2 seconds.


### Step 2 — Implement a callable wrapper

Use a named `RateLimitError` to distinguish overload from errors in the underlying function. Validate request cost *before* refilling or updating any state. A backwards-moving clock is an error, not a free token refill.

In [18]:
class RateLimitError(RuntimeError):
    pass

class TokenBucket:
    def __init__(self, fn, *, capacity, rate, clock):
        if not callable(fn) or not callable(clock):
            raise TypeError("fn and clock must be callable")
        if isinstance(capacity, bool) or not isinstance(capacity, (int, float)) or not isfinite(capacity) or capacity <= 0:
            raise ValueError("capacity must be finite and positive")
        if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not isfinite(rate) or rate < 0:
            raise ValueError("rate must be finite and nonnegative")
        self.fn, self.clock = fn, clock
        self.capacity, self.rate = float(capacity), float(rate)
        self.tokens = self.capacity
        self.last_time = clock()

    def __call__(self, *args, cost=1.0, **kwargs):
        if isinstance(cost, bool) or not isinstance(cost, (int, float)) or not isfinite(cost) or not 0 < cost <= self.capacity:
            raise ValueError("cost must be finite and between 0 and capacity")
        now = self.clock()
        elapsed = now - self.last_time
        if elapsed < 0:
            raise ValueError("clock moved backwards")
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_time = now
        if self.tokens < cost:
            raise RateLimitError("insufficient tokens")
        self.tokens -= cost
        return self.fn(*args, **kwargs)

### Step 3 — Prove rejection prevents work

Our fake operation appends to a list when it is actually executed. The log provides a direct observation of which requests got through.

In [19]:
processed = []
def record_job(job):
    processed.append(job)
    return job.upper()

limited = TokenBucket(record_job, capacity=3, rate=0.5, clock=rate_clock)
assert limited("alpha", cost=2) == "ALPHA"
assert limited("beta") == "BETA"
assert limited.tokens == 0
with expect_raises(RateLimitError):
    limited("rejected")
assert processed == ["alpha", "beta"]
print("Actual jobs executed:", processed)

Expected RateLimitError: insufficient tokens
Actual jobs executed: ['alpha', 'beta']


### Step 4 — Time-dependent and invalid-input edges

Two seconds replenish exactly one token. Rejections do not debit tokens, but elapsed time still updates the bucket. Validation should reject an impossible cost without changing the token balance.

In [20]:
rate_clock.advance(2)
assert limited("gamma") == "GAMMA"
assert limited.tokens == 0
snapshot = (limited.tokens, limited.last_time, list(processed))
with expect_raises(ValueError):
    limited("invalid", cost=4)
assert (limited.tokens, limited.last_time, processed) == snapshot
rate_clock.advance(20)
assert limited("delta", cost=3) == "DELTA"
assert abs(limited.tokens) < 1e-12
print("Problem 4: refill, capacity, and invalid costs passed")

Expected ValueError: cost must be finite and between 0 and capacity
Problem 4: refill, capacity, and invalid costs passed


### Trade-off — What does an operation failure cost?

This implementation consumes a token *before* invoking the operation. If the operation raises, the token stays consumed: admission capacity measures **attempts**, not successful completions. Returning the token after failure would be a different (valid) contract and would need explicit failure-handling logic.

---

# Problem 05 — Sliding-window event deduplication

**Challenge.** Create a callable event filter that recognizes repeated event IDs during a TTL window, expires old IDs, and keeps bounded state.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Separate the event ID from its payload

Consider repeated webhook deliveries. We want to accept the first delivery of an ID and reject later deliveries of the *same* ID until the ID expires. The payload can differ, but the ID is the deduplication identity. Our callable will return `True` when it accepts a new event and `False` when it recognizes a duplicate.

Use an `OrderedDict` of `{id: acceptance_time}`. Order tracks **acceptance times**, not last-seen times: duplicates do not refresh the expiry window.

In [21]:
event_clock = ManualClock(0)
print("At TTL=5, an event accepted at t=0 can be accepted again at t=5.")

At TTL=5, an event accepted at t=0 can be accepted again at t=5.


### Step 2 — Define storage and expiration precisely

Purge old entries **before** checking membership. Because accepted events are appended in time order, expired entries are at the front. If capacity is exceeded, evict the oldest accepted ID. Capacity eviction is not time expiration: an evicted ID may be accepted again sooner than TTL.

In [22]:
class EventDeduplicator:
    def __init__(self, *, ttl, capacity, clock):
        if isinstance(ttl, bool) or not isinstance(ttl, (int, float)) or not isfinite(ttl) or ttl <= 0:
            raise ValueError("ttl must be finite and positive")
        if type(capacity) is not int or capacity < 1:
            raise ValueError("capacity must be a positive integer")
        if not callable(clock):
            raise TypeError("clock must be callable")
        self.ttl = float(ttl)
        self.capacity = capacity
        self.clock = clock
        self._seen = OrderedDict()
        self._last_time = clock()
        self.accepted = 0
        self.duplicates = 0

    @property
    def active_ids(self):
        return tuple(self._seen)

    def __call__(self, event_id):
        hash(event_id)  # make the hashability requirement explicit
        now = self.clock()
        if now < self._last_time:
            raise ValueError("clock moved backwards")
        self._last_time = now
        while self._seen:
            first_id, timestamp = next(iter(self._seen.items()))
            if now - timestamp < self.ttl:
                break
            del self._seen[first_id]
        if event_id in self._seen:
            self.duplicates += 1
            return False
        if len(self._seen) == self.capacity:
            self._seen.popitem(last=False)
        self._seen[event_id] = now
        self.accepted += 1
        return True

### Step 3 — Duplicates must not extend the lifetime

Accept `A` at `t=0`; at `t=4`, a repeated `A` is rejected. At `t=5` it must be accepted again. If we had updated the timestamp on the duplicate, this important invariant would fail.

In [23]:
dedup = EventDeduplicator(ttl=5, capacity=2, clock=event_clock)
assert dedup("A") is True
event_clock.advance(4)
assert dedup("A") is False
assert dedup.active_ids == ("A",)
event_clock.advance(1)
assert dedup("A") is True
assert (dedup.accepted, dedup.duplicates) == (2, 1)
print("At expiry boundary:", dedup.active_ids)

At expiry boundary: ('A',)


### Step 4 — Capacity eviction and independent instances

At the same timestamp, adding `B` then `C` to a capacity-two filter evicts `A`. Another filter does not share the first filter's history.

In [24]:
assert dedup("B") is True
assert dedup("C") is True
assert dedup.active_ids == ("B", "C")
assert dedup("A") is True  # A was evicted due to capacity
assert dedup.active_ids == ("C", "A")
independent = EventDeduplicator(ttl=5, capacity=2, clock=event_clock)
assert independent("C") is True
with expect_raises(TypeError):
    dedup(["unhashable"])
print("Problem 5: expiry and bounded storage passed")

Expected TypeError: unhashable type: 'list'
Problem 5: expiry and bounded storage passed


### Production boundary

This is an in-memory, sequential filter. Restarting the process loses deduplication history; multiple processes need shared storage with atomic compare-and-insert for strong delivery guarantees. The cap creates a deliberate memory-versus-deduplication trade-off.

---

# Problem 06 — A callable function-contract enforcer

**Challenge.** Validate a wrapped callable’s individual arguments and its result using independently supplied predicate callables without silently changing the wrapped function’s signature rules.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Introduce validators as first-class callables

A validator can be a function, lambda, or callable instance; its *type* does not matter as long as it can be called. We will use a `positive_int` predicate and `nonempty_text` predicate. A predicate must return `True` to accept a value; other return values are treated as a rejection.

In [25]:
def positive_int(value):
    return type(value) is int and value > 0

def nonempty_text(value):
    return isinstance(value, str) and bool(value.strip())

assert positive_int(3)
assert not positive_int(True)  # bool is a subclass of int
assert not nonempty_text("   ")

### Step 2 — What happens to defaults and keyword calls?

`fn(3)` and `fn(number=3)` should be treated consistently. `inspect.signature(fn).bind(*args, **kwargs)` validates Python’s ordinary calling rules and maps argument names to values. `apply_defaults()` includes values omitted by the caller but supplied as defaults.

In this problem, validators are keyed by *ordinary named parameters* (`POSITIONAL_ONLY`, `POSITIONAL_OR_KEYWORD`, and `KEYWORD_ONLY`). We reject validators for unknown names or variadic `*args` / `**kwargs` parameters.

In [26]:
def example_operation(number, *, prefix="item"):
    return f"{prefix}:{number}"

sig = signature(example_operation)
bound = sig.bind(3)
bound.apply_defaults()
assert bound.arguments == {"number": 3, "prefix": "item"}
print("Normalized binding:", dict(bound.arguments))

Normalized binding: {'number': 3, 'prefix': 'item'}


### Step 3 — Implement the enforcer

We validate both the construction-time contract and each invocation. We deliberately leave exceptions raised by validators untouched: a buggy validator should be distinguishable from one that simply returned `False`. On a failed input check, the wrapped function must not run. On a failed output check, the underlying call has already happened, so side effects cannot be undone automatically.

In [27]:
class ContractViolation(ValueError):
    pass

class Contract:
    def __init__(self, fn, *, inputs=None, output=None):
        if not callable(fn):
            raise TypeError("fn must be callable")
        self.fn = fn
        self.signature = signature(fn)
        self.inputs = dict(inputs or {})
        for name, validator in self.inputs.items():
            parameter = self.signature.parameters.get(name)
            if parameter is None or parameter.kind in (parameter.VAR_POSITIONAL, parameter.VAR_KEYWORD):
                raise ValueError(f"unsupported parameter name: {name!r}")
            if not callable(validator):
                raise TypeError(f"validator for {name!r} is not callable")
        if output is not None and not callable(output):
            raise TypeError("output validator must be callable")
        self.output = output

    def __call__(self, *args, **kwargs):
        bound = self.signature.bind(*args, **kwargs)
        bound.apply_defaults()
        for name, validator in self.inputs.items():
            if name in bound.arguments and validator(bound.arguments[name]) is not True:
                raise ContractViolation(f"input {name!r} rejected")
        result = self.fn(*args, **kwargs)
        if self.output is not None and self.output(result) is not True:
            raise ContractViolation("result rejected")
        return result

### Step 4 — First use: preconditions and postconditions

A result validator is useful when interacting with an external component that promises a certain output type. Here the operation itself is simple; we focus on sequencing of checks.

In [28]:
executed = []
def format_order(quantity, *, label="order"):
    executed.append(quantity)
    return f"{label}#{quantity}"

checked_order = Contract(
    format_order,
    inputs={"quantity": positive_int, "label": nonempty_text},
    output=lambda result: isinstance(result, str) and "#" in result,
)
assert checked_order(2) == "order#2"
assert checked_order(quantity=4, label="box") == "box#4"
print("Calls executed:", executed)

Calls executed: [2, 4]


### Step 5 — Verify errors are not accidentally swallowed

A rejected argument should preserve the execution log, a malformed call should raise Python’s ordinary `TypeError`, and invalid configuration should fail at construction. We are exercising the entire boundary, not just the happy path.

In [29]:
snapshot = executed.copy()
with expect_raises(ContractViolation):
    checked_order(quantity=-1)
with expect_raises(ContractViolation):
    checked_order(3, label=" ")
with expect_raises(TypeError):
    checked_order()
assert executed == snapshot
with expect_raises(ValueError):
    Contract(format_order, inputs={"not_a_parameter": positive_int})

bad_result = Contract(lambda: None, output=lambda value: isinstance(value, str))
with expect_raises(ContractViolation):
    bad_result()
print("Problem 6: contracts and error paths passed")

Expected ContractViolation: input 'quantity' rejected
Expected ContractViolation: input 'label' rejected
Expected TypeError: missing a required argument: 'quantity'
Expected ValueError: unsupported parameter name: 'not_a_parameter'
Expected ContractViolation: result rejected
Problem 6: contracts and error paths passed


### Subtlety — A contract is not a general-purpose type checker

`inspect.signature` cannot inspect every possible built-in or extension callable, and a predicate may have side effects. This wrapper is intentionally for inspectable callables and simple pure validators; catching every exception and treating it as “invalid data” would hide programming errors.

---

# Problem 07 — Lazy dependency injection with per-call caching

**Challenge.** Build a callable resolver that creates an object graph on demand, constructs shared dependencies once per top-level resolution, and detects circular dependencies.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — What should `container("service")` mean?

A provider is a callable taking the resolver as its only argument. It can request other dependencies by calling the resolver: `lambda get: Service(get("database"))`. If two providers need `database` during one top-level request, build it **once**. A *separate* top-level request gets fresh objects.

The resolver must also detect a cycle such as `alpha → beta → alpha`; without cycle detection, the outcome would be an obscure recursion error.

In [30]:
class DependencyCycleError(RuntimeError):
    pass

print("Each provider will receive the same resolver inside a resolution session.")

Each provider will receive the same resolver inside a resolution session.


### Step 2 — Keep session state separate from registry state

A registry of providers persists. The cache and active dependency stack exist only while resolving one top-level request. Keeping these fields on the container itself is convenient for this **sequential, non-reentrant** exercise, but it must use `try/finally` to clean up even when a provider raises.

An `is not None` check cannot mark a dependency as cached because `None` may be a legitimate resolved value. Dictionary membership is the correct test.

In [31]:
class DependencyContainer:
    def __init__(self):
        self._providers = {}
        self._session = None
        self._active = None

    def register(self, name, provider):
        if not isinstance(name, str) or not name:
            raise ValueError("provider name must be a nonempty string")
        if not callable(provider):
            raise TypeError("provider must be callable")
        if name in self._providers:
            raise ValueError(f"already registered: {name}")
        self._providers[name] = provider

    def __call__(self, name):
        root = self._session is None
        if root:
            self._session, self._active = {}, []
        try:
            return self._resolve(name)
        finally:
            if root:
                self._session, self._active = None, None

    def _resolve(self, name):
        if name not in self._providers:
            raise KeyError(name)
        if name in self._active:
            path = " -> ".join([*self._active, name])
            raise DependencyCycleError(path)
        if name in self._session:
            return self._session[name]
        self._active.append(name)
        try:
            instance = self._providers[name](self)
            self._session[name] = instance
            return instance
        finally:
            self._active.pop()

### Step 3 — Resolve a diamond-shaped graph

Both `left` and `right` depend on the same `database`. We return dictionaries instead of opening real sockets. By inspecting the creation log and using `is`, we can verify the shared object was reused *within the session*.

In [32]:
creation_log = []
container = DependencyContainer()

def make_database(resolve):
    creation_log.append("database")
    return object()

container.register("database", make_database)
container.register("left", lambda get: {"db": get("database")})
container.register("right", lambda get: {"db": get("database")})
container.register("service", lambda get: (get("left"), get("right")))
left_service, right_service = container("service")
assert left_service["db"] is right_service["db"]
assert creation_log == ["database"]
print("Shared database within one call:", left_service["db"] is right_service["db"])

Shared database within one call: True


### Step 4 — Verify a new top-level call gets a fresh graph

A per-call cache is different from a global singleton. Calling `container("service")` again begins a new session and constructs a different database.

In [33]:
new_left, new_right = container("service")
assert new_left["db"] is new_right["db"]
assert new_left["db"] is not left_service["db"]
assert creation_log == ["database", "database"]
print("Independent top-level graph constructed")

Independent top-level graph constructed


### Step 5 — Detect cycles and clean up after failure

A failed resolution should not poison future requests. Once the cycle has raised, we should be able to resolve an unrelated provider immediately. For now, registering twice is prohibited so the cycle stays visible.

In [34]:
container.register("alpha", lambda get: get("beta"))
container.register("beta", lambda get: get("alpha"))
with expect_raises(DependencyCycleError):
    container("alpha")
with expect_raises(KeyError):
    container("missing")
assert container("database") is not None
with expect_raises(ValueError):
    container.register("database", lambda get: None)
print("Problem 7: shared dependencies, cycles, and cleanup passed")

Expected DependencyCycleError: alpha -> beta -> alpha
Expected KeyError: 'missing'
Expected ValueError: already registered: database
Problem 7: shared dependencies, cycles, and cleanup passed


### Scope warning

Because `_session` and `_active` live on the container, concurrent threads (or unrelated asynchronous tasks) must **not** share this implementation. A production container could use an explicit session object or task-local context to isolate graphs. The sequential contract here is intentionally testable and limited.

---

# Problem 08 — Transactional parser combinators

**Challenge.** Represent text parsers as callable objects that return a value and a new cursor without mutating the input, then compose parsers with sequencing and alternatives.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Parsing is a function of text and position

A parser will be invoked as `parser(text, position)` and return `ParseResult(value, next_position)`. If it cannot match, it raises `ParseError`. It does **not** mutate text or any shared cursor. That makes backtracking straightforward: an alternative parser can retry at the **original** position.

Our scope is a small deterministic grammar with literal tokens and digits, not a complete parser framework.

In [35]:
class ParseError(ValueError):
    pass

@dataclass(frozen=True)
class ParseResult:
    value: Any
    position: int

print("ParseResult stores a value and a cursor, not a mutable parser state.")

ParseResult stores a value and a cursor, not a mutable parser state.


### Step 2 — Start with a literal parser

A `Literal("cat")` must match exactly at the cursor. We reject invalid cursor types and bounds before trying to match. `startswith` makes the matching rule transparent.

In [36]:
class Literal:
    def __init__(self, token):
        if not isinstance(token, str) or not token:
            raise ValueError("token must be a nonempty string")
        self.token = token

    def __call__(self, text, position=0):
        if type(position) is not int or not 0 <= position <= len(text):
            raise ValueError("invalid position")
        if not text.startswith(self.token, position):
            raise ParseError(f"expected {self.token!r} at {position}")
        return ParseResult(self.token, position + len(self.token))

cat = Literal("cat")
assert cat("a cat", 2) == ParseResult("cat", 5)
with expect_raises(ParseError):
    cat("a dog", 2)

Expected ParseError: expected 'cat' at 2


### Step 3 — Add a digit parser that consumes one or more digits

Rather than relying on `str.isdigit()` (which accepts some Unicode digit characters), we explicitly choose ASCII digits `0..9` for this grammar. The parser returns an integer and stops at the first non-digit.

In [37]:
class Digits:
    def __call__(self, text, position=0):
        if type(position) is not int or not 0 <= position <= len(text):
            raise ValueError("invalid position")
        end = position
        while end < len(text) and "0" <= text[end] <= "9":
            end += 1
        if end == position:
            raise ParseError(f"expected digits at {position}")
        return ParseResult(int(text[position:end]), end)

number = Digits()
assert number("123 apples") == ParseResult(123, 3)
with expect_raises(ParseError):
    number("apples")

Expected ParseError: expected digits at 0


### Step 4 — Sequence parsers without losing cursor position

A sequence passes the first result’s cursor to the next parser and collects their values. If a later parser fails, no input state needs to be rolled back: we never changed any shared cursor in the first place.

In [38]:
class Sequence:
    def __init__(self, *parsers):
        if not parsers or not all(callable(p) for p in parsers):
            raise ValueError("provide at least one callable parser")
        self.parsers = tuple(parsers)

    def __call__(self, text, position=0):
        values = []
        for parser in self.parsers:
            result = parser(text, position)
            values.append(result.value)
            position = result.position
        return ParseResult(tuple(values), position)

assignment = Sequence(Literal("x="), Digits(), Literal(";"))
assert assignment("x=42; rest") == ParseResult(("x=", 42, ";"), 5)
print(assignment("x=42; rest"))

ParseResult(value=('x=', 42, ';'), position=5)


### Step 5 — Alternatives: retry from the original position

Our `Choice` catches `ParseError` only. A `TypeError` caused by buggy parser code should propagate instead of being mistaken for “this grammar does not match.” On failure, each branch receives exactly the same starting cursor.

In [39]:
class Choice:
    def __init__(self, *parsers):
        if not parsers or not all(callable(p) for p in parsers):
            raise ValueError("provide at least one callable parser")
        self.parsers = tuple(parsers)

    def __call__(self, text, position=0):
        for parser in self.parsers:
            try:
                return parser(text, position)
            except ParseError:
                continue
        raise ParseError(f"no alternative matched at {position}")

animal = Choice(Literal("cat"), Literal("dog"))
assert animal("dog!") == ParseResult("dog", 3)
assert animal("cat!") == ParseResult("cat", 3)
with expect_raises(ParseError):
    animal("bird!")

Expected ParseError: no alternative matched at 0


### Step 6 — A small grammar and its failure boundary

Combine `Choice` and `Sequence` to parse either `cat:3` or `dog:12`, while leaving any remaining suffix available for another parser. Notice that we have built a parser by **composing callable objects**, not by introducing a global cursor or a monolithic regular expression.

In [40]:
pet_count = Sequence(Choice(Literal("cat"), Literal("dog")), Literal(":"), Digits())
assert pet_count("dog:12!") == ParseResult(("dog", ":", 12), 6)
assert pet_count("cat:3") == ParseResult(("cat", ":", 3), 5)
with expect_raises(ParseError):
    pet_count("dog:!")
with expect_raises(ValueError):
    Literal("")
print("Problem 8: composition and backtracking passed")

Expected ParseError: expected digits at 4
Expected ValueError: token must be a nonempty string
Problem 8: composition and backtracking passed


### Design insight

When inputs and outputs are immutable values, backtracking and composition become much easier to reason about. A more complete parser would add richer error locations, full-input checks, repetition combinators, and careful handling of ambiguous grammars.

---

# Problem 09 — A callable event hub with safe mutation during delivery

**Challenge.** Implement an event dispatcher that allows any callable subscriber, returns unsubscribe functions, delivers from a snapshot, and isolates subscriber failures.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Decide what one invocation does

`hub(event)` will call all subscribers in their registration order. Each subscriber receives the same event. One subscriber might unsubscribe itself or subscribe another handler while receiving an event; neither should change the *current* dispatch list.

We will copy the ordered subscriber list at the start of each dispatch. The hub returns a list of exceptions instead of swallowing failures silently or abandoning later subscribers.

In [41]:
print("Callable subscribers include functions, bound methods, and objects with __call__.")

Callable subscribers include functions, bound methods, and objects with __call__.


### Step 2 — Store subscribers under unique numeric tokens

Removing a callback by equality can remove the wrong subscription when the same callback was registered twice. A unique token solves that ambiguity. The returned unsubscribe closure captures *one token*, and repeated unsubscription is harmless.

In [42]:
class EventHub:
    def __init__(self):
        self._subscribers = {}
        self._next_token = 0

    @property
    def subscriber_count(self):
        return len(self._subscribers)

    def subscribe(self, callback):
        if not callable(callback):
            raise TypeError("subscriber must be callable")
        token = self._next_token
        self._next_token += 1
        self._subscribers[token] = callback

        def unsubscribe():
            return self._subscribers.pop(token, None) is not None

        return unsubscribe

    def __call__(self, event):
        errors = []
        snapshot = tuple(self._subscribers.values())
        for callback in snapshot:
            try:
                callback(event)
            except Exception as exc:
                errors.append(exc)
        return errors

### Step 3 — Test a subscriber that unsubscribes itself

`self_unsubscribe` removes itself during dispatch. We still expect the remaining subscribers from the snapshot to run. On the *next* event, the removed handler should no longer be called.

In [43]:
hub = EventHub()
events = []

def first_handler(event):
    events.append(("first", event))
    unsubscribe_first()

def second_handler(event):
    events.append(("second", event))

unsubscribe_first = hub.subscribe(first_handler)
unsubscribe_second = hub.subscribe(second_handler)
assert hub("tick-1") == []
assert events == [("first", "tick-1"), ("second", "tick-1")]
assert hub("tick-2") == []
assert events[-1] == ("second", "tick-2")
assert hub.subscriber_count == 1
assert unsubscribe_first() is False
print("Dispatch log:", events)

Dispatch log: [('first', 'tick-1'), ('second', 'tick-1'), ('second', 'tick-2')]


### Step 4 — Failure isolation and mid-dispatch subscriptions

A callback raises, then another adds a new callback. The new subscriber should begin receiving events only from the next dispatch. We verify that the error remains available to the caller rather than disappearing.

In [44]:
def broken(event):
    raise RuntimeError(f"bad subscriber on {event}")

def register_late(event):
    if event == "first-pass":
        hub.subscribe(lambda e: events.append(("late", e)))

hub.subscribe(broken)
hub.subscribe(register_late)
errors = hub("first-pass")
assert len(errors) == 1 and isinstance(errors[0], RuntimeError)
assert ("late", "first-pass") not in events
errors = hub("second-pass")
assert len(errors) == 1
assert ("late", "second-pass") in events
assert unsubscribe_second() is True
with expect_raises(TypeError):
    hub.subscribe(42)
print("Problem 9: subscription snapshots and failure isolation passed")

Expected TypeError: subscriber must be callable
Problem 9: subscription snapshots and failure isolation passed


### Limitations

This implementation preserves ordinary *sequential* dispatch semantics. It does not promise thread-safe registration or asynchronous delivery. Snapshotting protects the current iteration from mutation; it does not make callbacks themselves side-effect-free.

---

# Problem 10 — A failure-aware batch writer

**Challenge.** Design a callable buffer that accepts one item per call, flushes at a threshold, and retains pending data if a sink fails.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Specify ownership of the batch

`writer(item)` appends an item and automatically flushes when there are `batch_size` pending items. `flush()` submits an **immutable tuple snapshot** to the sink and only clears pending data after the sink returns successfully. `close()` flushes remaining items and forbids future writes. Empty flushes should be no-ops.

We will assume a **non-reentrant**, sequential sink: it does not call the batch writer while handling a batch.

In [45]:
class BatchClosedError(RuntimeError):
    pass

print("If the sink raises, retain the batch so it can be retried.")

If the sink raises, retain the batch so it can be retried.


### Step 2 — Implement buffering and flushing

We choose `>= batch_size` rather than equality so the invariant is robust even if a future change appends multiple items at once. Returning the current pending count makes the callable easy to use interactively.

In [46]:
class BatchWriter:
    def __init__(self, sink, *, batch_size):
        if not callable(sink):
            raise TypeError("sink must be callable")
        if type(batch_size) is not int or batch_size < 1:
            raise ValueError("batch_size must be a positive integer")
        self.sink = sink
        self.batch_size = batch_size
        self._pending = []
        self.closed = False
        self.flushes = 0

    @property
    def pending(self):
        return tuple(self._pending)

    def __call__(self, item):
        if self.closed:
            raise BatchClosedError("writer is closed")
        self._pending.append(item)
        if len(self._pending) >= self.batch_size:
            self.flush()
        return len(self._pending)

    def flush(self):
        if not self._pending:
            return 0
        batch = tuple(self._pending)
        self.sink(batch)  # failure leaves the original list unchanged
        self._pending.clear()
        self.flushes += 1
        return len(batch)

    def close(self):
        if not self.closed:
            self.flush()  # a failure prevents closing and retains the batch
            self.closed = True

### Step 3 — Observe a normal auto-flush

A batch of three submissions should produce one sink invocation and leave no items pending. A partial batch remains in memory until explicitly flushed or closed.

In [47]:
written_batches = []
writer = BatchWriter(written_batches.append, batch_size=3)
assert writer("a") == 1
assert writer("b") == 2
assert writer("c") == 0
assert written_batches == [("a", "b", "c")]
assert writer.pending == () and writer.flushes == 1
assert writer("d") == 1
assert writer.flush() == 1
assert written_batches[-1] == ("d",)
assert writer.flush() == 0
print("Batches:", written_batches)

Batches: [('a', 'b', 'c'), ('d',)]


### Step 4 — Simulate a failing sink, then retry

The sink fails *before* accepting the batch. After a raised error, `pending` must still contain the item, and `closed` must remain false. We can then retry the flush. We deliberately do not guess whether a real remote system performed work before an exception.

In [48]:
class FailsOnceSink:
    def __init__(self):
        self.fail = True
        self.accepted = []

    def __call__(self, batch):
        if self.fail:
            self.fail = False
            raise OSError("simulated failure before commit")
        self.accepted.append(batch)

sink = FailsOnceSink()
resilient = BatchWriter(sink, batch_size=2)
assert resilient("x") == 1
with expect_raises(OSError):
    resilient("y")
assert resilient.pending == ("x", "y") and resilient.flushes == 0
resilient.close()  # close retries pending data successfully
assert sink.accepted == [("x", "y")]
assert resilient.closed and resilient.pending == ()
with expect_raises(BatchClosedError):
    resilient("z")
print("Problem 10: failure retention and close passed")

Expected OSError: simulated failure before commit
Expected BatchClosedError: writer is closed
Problem 10: failure retention and close passed


### Exactly-once delivery is outside this contract

Retaining data on failure gives *at-least-once retry potential*, not exactly-once delivery: a remote sink might commit the batch and then lose its acknowledgment. Production systems often need idempotency keys or a transaction protocol. Also, the sink must not recursively call this buffer during `flush()`.

---

# Problem 11 — Reproducible weighted sampling

**Challenge.** Implement a callable distribution sampler with strict weight validation, deterministic boundary behavior, independent random generators, and a reset facility.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Construct cumulative weight boundaries

For items `A`, `B`, `C` and weights `1, 2, 1`, cumulative boundaries are `1, 3, 4`. Draw `u` uniformly from `[0, total)`: select the first boundary strictly greater than `u`. Thus `u=1` selects `B`, and `u=3` selects `C`.

We will allow an injected zero-argument random function for deterministic boundary tests; otherwise each instance owns a `random.Random(seed)` generator.

In [49]:
from bisect import bisect_right
from random import Random

boundaries = [1.0, 3.0, 4.0]
assert [bisect_right(boundaries, x) for x in (0.0, 0.999, 1.0, 2.999, 3.0)] == [0, 0, 1, 1, 2]
print("Boundaries:", boundaries)

Boundaries: [1.0, 3.0, 4.0]


### Step 2 — Validate weights before calculating totals

Empty item lists, mismatched lengths, negative weights, all-zero weights, and non-finite weights should fail during construction. Zero-weight items are allowed but must never be sampled. For clarity, all weights are converted to floats; this is a teaching implementation for ordinary finite weights, not an arbitrary-precision probability library.

In [50]:
class WeightedSampler:
    def __init__(self, items, weights, *, seed=None, random_fn=None):
        items, weights = tuple(items), tuple(weights)
        if not items or len(items) != len(weights):
            raise ValueError("nonempty items and equally many weights required")
        for weight in weights:
            if isinstance(weight, bool) or not isinstance(weight, (int, float)) or not isfinite(weight) or weight < 0:
                raise ValueError("weights must be finite nonnegative numbers")
        self.items = items
        self._boundaries = []
        total = 0.0
        for weight in weights:
            total += weight
            self._boundaries.append(total)
        if not isfinite(total) or total <= 0:
            raise ValueError("total weight must be finite and positive")
        self.total = total
        self._seed = seed
        self._rng = Random(seed)
        if random_fn is not None and not callable(random_fn):
            raise TypeError("random_fn must be callable")
        self._external_random = random_fn

    def reset(self):
        if self._external_random is not None:
            raise RuntimeError("cannot reset an externally supplied random function")
        self._rng.seed(self._seed)

    def __call__(self):
        u = self._external_random() if self._external_random is not None else self._rng.random()
        if isinstance(u, bool) or not isinstance(u, (int, float)) or not isfinite(u) or not 0 <= u < 1:
            raise ValueError("random draw must lie in [0, 1)")
        draw = u * self.total
        # Clamp possible floating-point rounding at the upper edge.
        if draw >= self.total:
            import math
            draw = math.nextafter(self.total, 0.0)
        index = bisect_right(self._boundaries, draw)
        return self.items[index]

### Step 3 — Pin the boundary cases with a scripted random source

Our deterministic fake returns selected fractions of the total. This tests the exact boundary choice *without* using probabilistic assertions such as “roughly 25% of samples should be A.”

In [51]:
draws = iter([0.0, 0.25, 0.749, 0.75])
sampler_boundaries = WeightedSampler(["A", "B", "C"], [1, 2, 1], random_fn=lambda: next(draws))
assert [sampler_boundaries() for _ in range(4)] == ["A", "B", "B", "C"]
print("Boundary samples validated")

Boundary samples validated


### Step 4 — Show reproducibility and no global random-state pollution

Instances seeded identically should produce identical sequences. `reset()` restarts an instance’s own stream; it does not call the global `random.seed()` and therefore does not change randomness elsewhere in the process.

In [52]:
sampler_a = WeightedSampler(["x", "y"], [3, 1], seed=2026)
sampler_b = WeightedSampler(["x", "y"], [3, 1], seed=2026)
sequence = [sampler_a() for _ in range(20)]
assert sequence == [sampler_b() for _ in range(20)]
sampler_a.reset()
assert sequence == [sampler_a() for _ in range(20)]
with expect_raises(ValueError):
    WeightedSampler(["x"], [-1])
with expect_raises(ValueError):
    WeightedSampler(["x"], [0])
print("Problem 11: weights and reproducibility passed")

Expected ValueError: weights must be finite nonnegative numbers
Expected ValueError: total weight must be finite and positive
Problem 11: weights and reproducibility passed


### Further consideration

The sampler’s state lives in its own PRNG. Exposing a `seed` is useful for tests, but deterministic simulations should also record the Python runtime/environment when exact sequences matter across upgrades. A zero-weight item in the middle creates a repeated boundary; `bisect_right` skips that empty interval.

---

# Problem 12 — A composable predicate language with short-circuit rules

**Challenge.** Use callable predicates and overloaded `&`, `|`, and `~` to describe business rules while keeping Python’s built-in boolean semantics in mind.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — The operator trap: `and` cannot be customized

Python does not let classes overload the `and`, `or`, or `not` keywords. The operators `&`, `|`, and `~` **can** be overloaded via `__and__`, `__or__`, and `__invert__`. We will implement a lightweight rule language using those operators.

Parenthesize every predicate expression: `a & (b | c)`. Ordinary bitwise precedence may be surprising otherwise. Most importantly, we implement *short circuiting ourselves* inside `__call__`.

In [53]:
checks = []
def always_true(value):
    checks.append("true")
    return True

def should_not_run(value):
    checks.append("wrong")
    return False

print("The next implementation will not eagerly evaluate both sides.")

The next implementation will not eagerly evaluate both sides.


### Step 2 — A rule holds a predicate, not its already-computed result

`Rule(predicate)` validates the predicate at construction. `__call__(value)` returns `bool(...)` so the public result always has a true boolean type. Composition constructs *new* `Rule` instances that reference the original rules; it must never invoke the predicates during composition.

In [54]:
class Rule:
    def __init__(self, predicate, *, description="rule"):
        if not callable(predicate):
            raise TypeError("predicate must be callable")
        self.predicate = predicate
        self.description = description

    def __call__(self, value):
        return bool(self.predicate(value))

    def __and__(self, other):
        if not isinstance(other, Rule):
            return NotImplemented
        return Rule(lambda value: self(value) and other(value),
                    description=f"({self.description} & {other.description})")

    def __or__(self, other):
        if not isinstance(other, Rule):
            return NotImplemented
        return Rule(lambda value: self(value) or other(value),
                    description=f"({self.description} | {other.description})")

    def __invert__(self):
        return Rule(lambda value: not self(value),
                    description=f"~({self.description})")

    def __repr__(self):
        return f"Rule({self.description})"

### Step 3 — Prove our logical operators short-circuit

In the `OR` case, a true left side prevents the right side from running. For `AND`, a false left side prevents the right side from running. These are behavioral tests, not just checks that the final truth value is correct.

In [55]:
checks.clear()
rule_or = Rule(always_true, description="yes") | Rule(should_not_run, description="no")
assert rule_or("anything") is True
assert checks == ["true"]
checks.clear()
rule_and = Rule(lambda value: False, description="false") & Rule(should_not_run)
assert rule_and("anything") is False
assert checks == []
print("Short-circuit checks passed")

Short-circuit checks passed


### Step 4 — Solve a concrete rule-composition problem

Accept an order if its amount is positive **and** either it has a valid coupon or it belongs to a preferred customer. Reject orders where any of those requirements fail. Each elementary rule stays independently testable and reusable.

In [56]:
positive_amount = Rule(lambda order: order["amount"] > 0, description="positive amount")
valid_coupon = Rule(lambda order: order["coupon"] == "SAVE", description="valid coupon")
preferred = Rule(lambda order: order["preferred"] is True, description="preferred")
accept_order = positive_amount & (valid_coupon | preferred)

orders = [
    {"amount": 10, "coupon": "SAVE", "preferred": False},
    {"amount": 20, "coupon": "NO", "preferred": True},
    {"amount": 20, "coupon": "NO", "preferred": False},
    {"amount": 0, "coupon": "SAVE", "preferred": True},
]
assert [accept_order(order) for order in orders] == [True, True, False, False]
assert (~preferred)(orders[0]) is True
print("Rule:", accept_order)

Rule: Rule((positive amount & (valid coupon | preferred)))


### Step 5 — Input validation and a precedence reminder

Passing a non-Rule to `&` returns `NotImplemented`, allowing Python to try the other operand’s reflected method before raising a `TypeError`. By contrast, the predicate itself can raise `KeyError` for a malformed order; this exercise deliberately propagates that error rather than silently rejecting it.

In [57]:
with expect_raises(TypeError):
    Rule(99)
with expect_raises(TypeError):
    _ = accept_order & 12
with expect_raises(KeyError):
    accept_order({"amount": 5})
print("Problem 12: lazy composition and invalid inputs passed")

Expected TypeError: predicate must be callable
Expected TypeError: unsupported operand type(s) for &: 'Rule' and 'int'
Expected KeyError: 'coupon'
Problem 12: lazy composition and invalid inputs passed


### Design insight

These rule objects resemble a small domain-specific language: composition happens first, evaluation happens later. Because `__call__` is the common interface, the same engine accepts ordinary functions and other callable instances. Keep predicates pure when consistent repeated evaluation matters.

---

# Problem 13 — Capstone: a callable checkout workflow

**Challenge.** Combine independently testable callables into an orchestrator that validates orders, admits work under a token budget, and records successful and failed attempts without hiding errors.

We will **experiment → reason → implement → check edge cases → reflect**, rather than jump directly to the finished class.

### Step 1 — Choose a precise end-to-end contract

The earlier objects can be reused, but their order matters. We want a workflow called as `checkout(order, *, cost=1)` that does three things:

1. Validate input using a `Rule`. A rejected order raises `OrderRejectedError` *before* consuming rate-limit tokens.
2. If valid, call a `TokenBucket`-protected charge function. When rate-limited, do not charge.
3. Record each attempted checkout in an audit log as `accepted`, `rejected`, or `failed`, including failures caused by the underlying charge function.

We will specify the audit representation as immutable tuples rather than returning a mutable internal log.

In [58]:
class OrderRejectedError(ValueError):
    pass

checkout_clock = ManualClock(0)
charged_orders = []

def charge(order):
    if order.get("simulate_failure", False):
        raise OSError("payment provider unavailable")
    charged_orders.append(order["id"])
    return f"receipt:{order['id']}"

charge_limited = TokenBucket(charge, capacity=2, rate=1, clock=checkout_clock)

### Step 2 — Write the thin orchestration layer

The orchestrator does not copy the token-bucket algorithm or the rule engine. It accepts a callable validator and callable executor as dependencies, which makes unit testing straightforward.

We record validation rejection separately because the executor must not be invoked in that case. We use `try/except Exception` around the executor so the original exception still propagates with a bare `raise`.

In [59]:
class CheckoutWorkflow:
    def __init__(self, validator, executor):
        if not callable(validator) or not callable(executor):
            raise TypeError("validator and executor must be callable")
        self.validator = validator
        self.executor = executor
        self._audit = []

    @property
    def audit(self):
        return tuple(self._audit)

    def __call__(self, order, *, cost=1):
        order_id = order.get("id")
        try:
            valid = self.validator(order)
        except Exception:
            self._audit.append((order_id, "failed"))
            raise
        if not valid:
            self._audit.append((order_id, "rejected"))
            raise OrderRejectedError("order does not satisfy the checkout rules")
        try:
            receipt = self.executor(order, cost=cost)
        except Exception:
            self._audit.append((order_id, "failed"))
            raise
        self._audit.append((order_id, "accepted"))
        return receipt

### Step 3 — Compose a rule specifically for checkout

A valid order must have a positive amount and a nonempty string ID. We use explicit key checks so missing data produces a controlled *rejection* rather than a surprise `KeyError` in this example. In real applications, a typed request object or input schema could make the boundary stronger.

In [60]:
has_id = Rule(lambda order: isinstance(order.get("id"), str) and bool(order["id"].strip()), description="has ID")
valid_amount = Rule(lambda order: type(order.get("amount")) in (int, float) and isfinite(order["amount"]) and order["amount"] > 0, description="valid amount")
checkout_rule = has_id & valid_amount
checkout = CheckoutWorkflow(checkout_rule, charge_limited)
assert checkout({"id": "O-1", "amount": 20}) == "receipt:O-1"
assert charged_orders == ["O-1"]
assert checkout.audit == (("O-1", "accepted"),)
print("First receipt issued")

First receipt issued


### Step 4 — A rejected order must not consume a token

Capture the token balance before validation fails. Then try another good order. It should still be admitted because the rejected order never reached the limiter.

In [61]:
balance = charge_limited.tokens
with expect_raises(OrderRejectedError):
    checkout({"id": "BAD", "amount": -5})
assert charge_limited.tokens == balance
assert checkout({"id": "O-2", "amount": 8}) == "receipt:O-2"
assert charged_orders == ["O-1", "O-2"]
assert charge_limited.tokens == 0
print("Rejection did not spend capacity")

Expected OrderRejectedError: order does not satisfy the checkout rules
Rejection did not spend capacity


### Step 5 — Reject overload before charging

The third valid order arrives while the token balance is zero. It is a *failed checkout attempt* from the workflow’s point of view, but the underlying charge function must never see it.

In [62]:
with expect_raises(RateLimitError):
    checkout({"id": "O-3", "amount": 1})
assert charged_orders == ["O-1", "O-2"]
assert checkout.audit[-1] == ("O-3", "failed")
checkout_clock.advance(1)
assert checkout({"id": "O-3", "amount": 1}) == "receipt:O-3"
assert charged_orders[-1] == "O-3"
print("Overload prevention and refill verified")

Expected RateLimitError: insufficient tokens
Overload prevention and refill verified


### Step 6 — The underlying service fails: preserve the exception and record it

After another second, a fourth order reaches the charge function but fails. We expect the original `OSError`, a failed audit entry, and no new successful charge. Because our token bucket measures attempts, this failure still consumes one token.

In [63]:
checkout_clock.advance(1)
before_charges = charged_orders.copy()
with expect_raises(OSError):
    checkout({"id": "O-4", "amount": 50, "simulate_failure": True})
assert charged_orders == before_charges
assert checkout.audit[-1] == ("O-4", "failed")
assert charge_limited.tokens == 0
assert tuple(status for _, status in checkout.audit) == (
    "accepted", "rejected", "accepted", "failed", "accepted", "failed"
)
print("Problem 13: end-to-end workflow and all failure paths passed")

Expected OSError: payment provider unavailable
Problem 13: end-to-end workflow and all failure paths passed


### Step 7 — Reason about design choices

This capstone demonstrates why `__call__` is useful beyond toy examples: a `Rule`, a `TokenBucket`, and a `CheckoutWorkflow` all share a function-like interface while each owns its own state and responsibility.

The audit log here is in memory and is **not** a financial ledger. A production checkout needs transaction boundaries, idempotency for retries, persistence, concurrency controls, and secure payment handling. We intentionally do not claim this demonstration provides those guarantees.

---

# Final review — A callable-object design checklist

Before introducing `__call__`, ask these questions:

1. **Contract:** What arguments are accepted? What does calling the object return? Which exceptions can escape?
2. **State:** What changes on success, rejection, or failure? Is state per instance or accidentally shared?
3. **Determinism:** Can time, randomness, I/O, and other dependencies be injected for tests?
4. **Safety:** Does validation happen before side effects? What is the behavior at exact boundaries?
5. **Composition:** Could this object accept other callables instead of hard-coding dependencies?
6. **Concurrency:** Is the object intended for sequential calls, threads, or async tasks? Never imply concurrency safety without implementing it.
7. **Introspection:** Would a plain function be simpler? Use a class when persistent state or additional methods/properties make the interface clearer.

**Recommended practice:** Re-run all cells from a fresh kernel. Change one specification (for example, TTL expiry at `>` instead of `>=`), predict which assertion fails, and only then edit the implementation. This reinforces the difference between an example that happens to work and an interface with a tested contract.

In [64]:
print("Workshop complete: 13 worked problems; all final assertions should pass.")

Workshop complete: 13 worked problems; all final assertions should pass.
